In [0]:
%pip install -U -qqqq mlflow langchain langgraph==0.3.4 databricks-langchain pydantic databricks-agents unitycatalog-langchain[databricks] uv
dbutils.library.restartPython()

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
%%writefile agent.py
from typing import Any, Generator, Optional, Sequence, Union

import mlflow
from databricks_langchain import (
    ChatDatabricks,
    VectorSearchRetrieverTool,
    DatabricksFunctionClient,
    UCFunctionToolkit,
    set_uc_function_client,
)
from langchain_core.language_models import LanguageModelLike
from langchain_core.runnables import RunnableConfig, RunnableLambda
from langchain_core.tools import BaseTool
from langgraph.graph import END, StateGraph
from langgraph.graph.graph import CompiledGraph
from langgraph.graph.state import CompiledStateGraph
from langgraph.prebuilt.tool_node import ToolNode
from mlflow.langchain.chat_agent_langgraph import ChatAgentState, ChatAgentToolNode
from mlflow.pyfunc import ChatAgent
from mlflow.types.agent import (
    ChatAgentChunk,
    ChatAgentMessage,
    ChatAgentResponse,
    ChatContext,
)

mlflow.langchain.autolog()

client = DatabricksFunctionClient()
set_uc_function_client(client)

############################################
# Define your LLM endpoint and system prompt
############################################
LLM_ENDPOINT_NAME = "databricks-meta-llama-3-3-70b-instruct"
llm = ChatDatabricks(endpoint=LLM_ENDPOINT_NAME)

system_prompt = """When you receive a query starting with 'PagerDuty:' followed by an alert summary, generate a well-structured JSON response with the following properties:

summary: A concise 2-3 sentence explanation of the root cause and recommended solution.

actions: An array of one or more executable steps (CLI commands, API calls, or procedural checks) to resolve the alert. Each action will
have a risk property associated with it that can have any value in ["high", "medium", "low"]. Commands that can alter system state like reboot will have high risk, something like stopping a service can have medium risk and simple commands can have low risk. Categorise them accordingly. 

followup: An array of suggestins that could be followup actions.

only return the json string.
Example:

json
{
  "summary": "The alert indicates high CPU utilization in the frontend service. Scale the pods and check for runaway processes.",
  "actions": [
    {
        "command": "kubectl scale deployment/frontend --replicas=4 -n production",
        "risk": "low"
    },
    {
        "command": "kubectl top pods -n production --sort-by=cpu",
        "risk": "low"
    },
    {
        "command": "ssh <user>@<node_ip> 'ps aux | grep <service_name> | sort -k3nr | head -5'",
        "risk": "low"
    },
    
  ],
  "followup": [
    "Monitor the CPU usage after scaling to ensure it returns to normal levels.",
    "Investigate any pods or processes consistently consuming high CPU and review their logs for errors or unusual activity.",
    "Consider setting up resource limits and requests in the deployment YAML to prevent similar issues in the future.",
    "If high CPU usage persists, review recent code changes or deployments for potential performance regressions.",
    "Document the incident and actions taken for future reference and post-incident analysis."
  ]

}

When you receive a query starting with 'Logs:' followed by logs, return a json object with one property named 'summary'. The summary property will have a value that is a concise summary of the given logs. The given logs should be summarized without missing any important technical details.
"""

###############################################################################
## Define tools for your agent, enabling it to retrieve data or take actions
## beyond text generation
## To create and see usage examples of more tools, see
## https://docs.databricks.com/generative-ai/agent-framework/agent-tool.html
###############################################################################
tools = []

# You can use UDFs in Unity Catalog as agent tools
uc_tool_names = ["sre.dev.srecet_docs_vector_search"]
uc_toolkit = UCFunctionToolkit(function_names=uc_tool_names)
tools.extend(uc_toolkit.tools)


# # (Optional) Use Databricks vector search indexes as tools
# # See https://docs.databricks.com/generative-ai/agent-framework/unstructured-retrieval-tools.html
# # for details
#
# # TODO: Add vector search indexes as tools or delete this block
# vector_search_tools = [
#         VectorSearchRetrieverTool(
#         index_name="",
#         # filters="..."
#     )
# ]
# tools.extend(vector_search_tools)


#####################
## Define agent logic
#####################


def create_tool_calling_agent(
    model: LanguageModelLike,
    tools: Union[Sequence[BaseTool], ToolNode],
    system_prompt: Optional[str] = None,
) -> CompiledGraph:
    model = model.bind_tools(tools)

    # Define the function that determines which node to go to
    def should_continue(state: ChatAgentState):
        messages = state["messages"]
        last_message = messages[-1]
        # If there are function calls, continue. else, end
        if last_message.get("tool_calls"):
            return "continue"
        else:
            return "end"

    if system_prompt:
        preprocessor = RunnableLambda(
            lambda state: [{"role": "system", "content": system_prompt}]
            + state["messages"]
        )
    else:
        preprocessor = RunnableLambda(lambda state: state["messages"])
    model_runnable = preprocessor | model

    def call_model(
        state: ChatAgentState,
        config: RunnableConfig,
    ):
        response = model_runnable.invoke(state, config)

        return {"messages": [response]}

    workflow = StateGraph(ChatAgentState)

    workflow.add_node("agent", RunnableLambda(call_model))
    workflow.add_node("tools", ChatAgentToolNode(tools))

    workflow.set_entry_point("agent")
    workflow.add_conditional_edges(
        "agent",
        should_continue,
        {
            "continue": "tools",
            "end": END,
        },
    )
    workflow.add_edge("tools", "agent")

    return workflow.compile()


class LangGraphChatAgent(ChatAgent):
    def __init__(self, agent: CompiledStateGraph):
        self.agent = agent

    def predict(
        self,
        messages: list[ChatAgentMessage],
        context: Optional[ChatContext] = None,
        custom_inputs: Optional[dict[str, Any]] = None,
    ) -> ChatAgentResponse:
        request = {"messages": self._convert_messages_to_dict(messages)}

        messages = []
        for event in self.agent.stream(request, stream_mode="updates"):
            for node_data in event.values():
                messages.extend(
                    ChatAgentMessage(**msg) for msg in node_data.get("messages", [])
                )
        return ChatAgentResponse(messages=messages)

    def predict_stream(
        self,
        messages: list[ChatAgentMessage],
        context: Optional[ChatContext] = None,
        custom_inputs: Optional[dict[str, Any]] = None,
    ) -> Generator[ChatAgentChunk, None, None]:
        request = {"messages": self._convert_messages_to_dict(messages)}
        for event in self.agent.stream(request, stream_mode="updates"):
            for node_data in event.values():
                yield from (
                    ChatAgentChunk(**{"delta": msg}) for msg in node_data["messages"]
                )


# Create the agent object, and specify it as the agent object to use when
# loading the agent back for inference via mlflow.models.set_model()
agent = create_tool_calling_agent(llm, tools, system_prompt)
AGENT = LangGraphChatAgent(agent)
mlflow.models.set_model(AGENT)

In [0]:
dbutils.library.restartPython()

In [0]:
from agent import AGENT

AGENT.predict({"messages": [{"role": "user", "content": "PagerDuty: Status: Acknowledged\nUrgency: High\nTitle: Unable to write to temporary directory\nTime (UTC): 2025-04-27 04:35:54\nSummary of the Issue: The system encountered an issue where it was unable to write to a temporary directory, which may indicate a permissions issue or lack of available space.\nEnvironment/Cluster: Production (inferred based on urgency)\nHost/Node Name: Not specified\nError Type: Filesystem access issue (possible permissions or disk space)\nImpacted Component: Temporary directory\nTime: N/A"}]})

In [0]:
for event in AGENT.predict_stream(
    {"messages": [{"role": "user", "content": "Logs: [8:06 PM, 4/28/2025] Annu Jolly: journalctl logs (system level) \n Apr 28 07:34:45 node-01 kernel: \n EXT4-fs warning (device sda1): ext4_dx_add_entry:2376: Directory index full! \n Apr 28 07:34:47 node-01 kernel: blk_update_request: I/O error, dev sda, \n sector 104857600 op 0x0:(READ) flags 0x0 phys_seg 8 prio class 0 \n Apr 28 07:34:48 node-01 systemd[1]: Starting Cleanup of Temporary Directories... \n Apr 28 07:34:50 node-01 systemd-tmpfiles[3456]: Failed to create file /var/tmp/somefile: No space left on device \n Apr 28 07:35:00 node-01 kernel: EXT4-fs error (device sda1): ext4_find_entry:1524: inode #655360: comm containerd: reading directory lblock 0 \n [8:06 PM, 4/28/2025] Annu Jolly: k logs \n Apr 28 07:35:08 node-01 kubelet[1234]: W0428 07:35:08.123456    1234 container_manager_linux.go:523] Running out of disk space available_bytes=10485760 capacity_bytes=107374182400 \n Apr 28 07:35:10 node-01 kubelet[1234]: E0428 07:35:10.654321    1234 pod_workers.go:191] Error syncing pod pod=default/test-pod err=failed to start container: no space left on device \n Apr 28 07:35:12 node-01 kubelet[1234]: W0428 07:35:12.112233    1234 kubelet_node_status.go:1112] Node not ready: insufficient disk space \nApr 28 07:35:15 node-01 kubelet[1234]: E0428 07:35:15.987654    1234 eviction_manager.go:263] Eviction manager: could not free enough space available=10Mi threshold=50Mi \n Apr 28 07:35:18 node-01 kubelet[1234]: E0428 07:35:18.765432    1234 status_manager.go:157] Failed to update pod status in apiserver pod=default/critical-app err=rpc error: code = ResourceExhausted desc = no space left on device Apr 28 07:35:21 node-01 kubelet[1234]: E0428 07:35:21.456789    1234 container_manager_linux.go:593] Failed to update cgroup resource resource=ephemeral-storage error=no space left on device Apr 28 07:35:25 node-01 kubelet[1234]: W0428 07:35:25.987654    1234 kubelet_node_status.go:1112] Node condition updated old=Ready new=NotReady reason=KubeletHasDiskPressure Apr 28 07:35:30 node-01 kubelet[1234]: E0428 07:35:30.123123    1234 volume_manager.go:351] Failed to detach volume volumeName=pvc-xyz err=cannot unmount volume, no space left on device"}]}
):
    print(event, "-----------\n")

In [0]:
# Determine Databricks resources to specify for automatic auth passthrough at deployment time
import mlflow
from agent import tools, LLM_ENDPOINT_NAME
from databricks_langchain import VectorSearchRetrieverTool
from mlflow.models.resources import DatabricksFunction, DatabricksServingEndpoint
from unitycatalog.ai.langchain.toolkit import UnityCatalogTool

# TODO: Manually include underlying resources if needed. See the TODO in the markdown above for more information.
resources = [DatabricksServingEndpoint(endpoint_name=LLM_ENDPOINT_NAME)]
for tool in tools:
    if isinstance(tool, VectorSearchRetrieverTool):
        resources.extend(tool.resources)
    elif isinstance(tool, UnityCatalogTool):
        resources.append(DatabricksFunction(function_name=tool.uc_function_name))

input_example = {
    "messages": [
        {
            "role": "user",
            "content": "PagerDuty: Status: Acknowledged\nUrgency: High\nTitle: Unable to write to temporary directory\nTime (UTC): 2025-04-27 04:35:54\nSummary of the Issue: The system encountered an issue where it was unable to write to a temporary directory, which may indicate a permissions issue or lack of available space.\nEnvironment/Cluster: Production (inferred based on urgency)\nHost/Node Name: Not specified\nError Type: Filesystem access issue (possible permissions or disk space)\nImpacted Component: Temporary directory\nTime: N/A"
        },
        {
            "role": "user",
            "content": "Logs: [8:06 PM, 4/28/2025] Annu Jolly: journalctl logs (system level) \n Apr 28 07:34:45 node-01 kernel: \n EXT4-fs warning (device sda1): ext4_dx_add_entry:2376: Directory index full! \n Apr 28 07:34:47 node-01 kernel: blk_update_request: I/O error, dev sda, \n sector 104857600 op 0x0:(READ) flags 0x0 phys_seg 8 prio class 0 \n Apr 28 07:34:48 node-01 systemd[1]: Starting Cleanup of Temporary Directories... \n Apr 28 07:34:50 node-01 systemd-tmpfiles[3456]: Failed to create file /var/tmp/somefile: No space left on device \n Apr 28 07:35:00 node-01 kernel: EXT4-fs error (device sda1): ext4_find_entry:1524: inode #655360: comm containerd: reading directory lblock 0 \n [8:06 PM, 4/28/2025] Annu Jolly: k logs \n Apr 28 07:35:08 node-01 kubelet[1234]: W0428 07:35:08.123456    1234 container_manager_linux.go:523] Running out of disk space available_bytes=10485760 capacity_bytes=107374182400 \n Apr 28 07:35:10 node-01 kubelet[1234]: E0428 07:35:10.654321    1234 pod_workers.go:191] Error syncing pod pod=default/test-pod err=failed to start container: no space left on device \n Apr 28 07:35:12 node-01 kubelet[1234]: W0428 07:35:12.112233    1234 kubelet_node_status.go:1112] Node not ready: insufficient disk space \nApr 28 07:35:15 node-01 kubelet[1234]: E0428 07:35:15.987654    1234 eviction_manager.go:263] Eviction manager: could not free enough space available=10Mi threshold=50Mi \n Apr 28 07:35:18 node-01 kubelet[1234]: E0428 07:35:18.765432    1234 status_manager.go:157] Failed to update pod status in apiserver pod=default/critical-app err=rpc error: code = ResourceExhausted desc = no space left on device Apr 28 07:35:21 node-01 kubelet[1234]: E0428 07:35:21.456789    1234 container_manager_linux.go:593] Failed to update cgroup resource resource=ephemeral-storage error=no space left on device Apr 28 07:35:25 node-01 kubelet[1234]: W0428 07:35:25.987654    1234 kubelet_node_status.go:1112] Node condition updated old=Ready new=NotReady reason=KubeletHasDiskPressure Apr 28 07:35:30 node-01 kubelet[1234]: E0428 07:35:30.123123    1234 volume_manager.go:351] Failed to detach volume volumeName=pvc-xyz err=cannot unmount volume, no space left on device"
        }
    ]
}

with mlflow.start_run():
    logged_agent_info = mlflow.pyfunc.log_model(
        artifact_path="agent",
        python_model="agent.py",
        input_example=input_example,
        resources=resources,
        extra_pip_requirements=[
            "databricks-connect"
        ]
    )

In [0]:
mlflow.models.predict(
    model_uri=f"runs:/{logged_agent_info.run_id}/agent",
    input_data={"messages": [{"role": "user", "content": "PagerDuty: Status: Acknowledged\nUrgency: High\nTitle: Unable to write to temporary directory\nTime (UTC): 2025-04-27 04:35:54\nSummary of the Issue: The system encountered an issue where it was unable to write to a temporary directory, which may indicate a permissions issue or lack of available space.\nEnvironment/Cluster: Production (inferred based on urgency)\nHost/Node Name: Not specified\nError Type: Filesystem access issue (possible permissions or disk space)\nImpacted Component: Temporary directory\nTime: N/A"}]},
    env_manager="uv",
)

In [0]:
mlflow.set_registry_uri("databricks-uc")

# TODO: define the catalog, schema, and model name for your UC model
catalog = "sre"
schema = "dev"
model_name = "srecet"
UC_MODEL_NAME = f"{catalog}.{schema}.{model_name}"

# register the model to UC
uc_registered_model_info = mlflow.register_model(
    model_uri=logged_agent_info.model_uri, name=UC_MODEL_NAME
)

In [0]:
from databricks import agents
agents.deploy(UC_MODEL_NAME, uc_registered_model_info.version, tags = {"endpointSource": "playground"})